# AED multiclasse v2 otimizado — avaliação realista e TFLite

Esta versão corrige os principais problemas do notebook anterior:

- nenhuma janela é escolhida com base na classe verdadeira;
- todas as classes usam a mesma janela deslizante uniforme;
- o ator do ASVP-ESD permanece no mesmo split;
- várias gravações do mesmo ator são preservadas;
- augmentation somente no treino;
- avaliação por janela e por gravação;
- limiares calibrados apenas na validação;
- estimativa de falsos alertas por hora;
- validação dos modelos TFLite float32, float16 e INT8;
- VAD separado do AED.

As classes continuam:

`ambiente_normal`, `grito_panico`, `choro_distress`, `voz_agressiva`,
`impacto`, `vidro_quebrando` e `tiro`.

## Antes de executar

No Colab:

```text
Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU
```

A métrica principal será o **F1 macro por gravação**, não apenas a acurácia por janela.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install                 librosa==0.11.0                 soundfile                 scikit-learn                 pandas                 matplotlib                 seaborn                 requests                 tqdm                 pyyaml                 webrtcvad-wheels \
                kagglehub

In [ ]:
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any
from IPython.display import Audio, display
from tqdm.auto import tqdm

import hashlib
import io
import json
import math
import random
import shutil
import subprocess
import time
import zipfile

import kagglehub
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import soundfile as sf
import tensorflow as tf
import webrtcvad

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
)
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

## 1. Configuração geral

In [ ]:
@dataclass(frozen=True)
class ConfiguracaoAED:
    taxa_amostragem: int = 16000
    duracao_janela: float = 2.0

    # A mesma regra de janelas será usada para todas as classes.
    passo_janela_dataset: float = 1.0
    passo_janela_inferencia: float = 0.5
    max_janelas_por_fonte: int = 4

    n_fft: int = 512
    hop_length: int = 256
    n_mels: int = 64
    n_mfcc: int = 40
    fmin: int = 40
    fmax: int = 7600

    max_grupos_por_classe: int = 180
    max_arquivos_por_grupo_classe: int = 8
    copias_augmentation_treino: int = 1

    proporcao_treino: float = 0.70
    proporcao_validacao: float = 0.15
    proporcao_teste: float = 0.15
    tentativas_split: int = 2500

    epocas: int = 60
    tamanho_lote: int = 32
    paciencia: int = 10
    taxa_aprendizado: float = 0.001
    semente: int = 42

    precisao_minima_limiar: float = 0.55
    limiar_minimo: float = 0.30
    limiar_maximo: float = 0.90
    minimo_eventos_consecutivos: int = 2
    suavizacao_janelas: int = 3

    usar_esc50: bool = True
    usar_tut_rare: bool = True
    usar_freesound: bool = False

    # True força a criação de features novas e evita reutilizar
    # o cache incorreto do notebook anterior.
    reprocessar_features: bool = True


CFG = ConfiguracaoAED()

CLASSES = [
    "ambiente_normal",
    "grito_panico",
    "choro_distress",
    "voz_agressiva",
    "impacto",
    "vidro_quebrando",
    "tiro",
]

ID_CLASSE = {
    nome: indice
    for indice, nome in enumerate(CLASSES)
}

CLASSES_EVENTO = [
    classe
    for classe in CLASSES
    if classe != "ambiente_normal"
]

RAIZ = Path("/content/aed_multiclasse_v2")
DADOS = RAIZ / "datasets"
ESC50 = DADOS / "esc50"
TUT = DADOS / "tut_rare"
FREESOUND = DADOS / "freesound"
PROCESSADO = RAIZ / "processado"
MODELOS = RAIZ / "modelos"
RESULTADOS = RAIZ / "resultados"
ENTREGA = RAIZ / "entrega"

for pasta in (
    DADOS,
    ESC50,
    TUT,
    FREESOUND,
    PROCESSADO,
    MODELOS,
    RESULTADOS,
    ENTREGA,
):
    pasta.mkdir(parents=True, exist_ok=True)

random.seed(CFG.semente)
np.random.seed(CFG.semente)
tf.random.set_seed(CFG.semente)

display(pd.DataFrame([asdict(CFG)]).T.rename(columns={0: "valor"}))
print("Classes:", ID_CLASSE)

## 2. Baixar ESC-50 e TUT Rare Sound Events

O notebook baixa somente o arquivo de eventos isolados do TUT, não os 17,5 GB completos.

O TUT fornece:

- `baby cry`
- `glass break`
- `gun shot`

O ESC-50 fornece:

- `crying_baby`
- `glass_breaking`
- `door_wood_knock`
- diversas classes normais e negativos difíceis, incluindo `fireworks`, `thunderstorm` e `siren`.

In [ ]:
URL_ESC50 = (
    "https://github.com/karolpiczak/ESC-50/"
    "archive/refs/heads/master.zip"
)

URL_TUT_EVENTOS = (
    "https://zenodo.org/records/401395/files/"
    "TUT-rare-sound-events-2017-development."
    "source_data_events.zip?download=1"
)


def baixar(
    url: str,
    destino: Path,
) -> None:
    if destino.exists() and destino.stat().st_size > 0:
        print("Já existe:", destino)
        return

    with requests.get(
        url,
        stream=True,
        timeout=60,
    ) as resposta:
        resposta.raise_for_status()
        total = int(
            resposta.headers.get(
                "content-length",
                0,
            )
        )

        with destino.open("wb") as arquivo:
            with tqdm(
                total=total,
                unit="B",
                unit_scale=True,
                desc=destino.name,
            ) as barra:
                for bloco in resposta.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if not bloco:
                        continue

                    arquivo.write(bloco)
                    barra.update(len(bloco))


def extrair_zip(
    arquivo_zip: Path,
    destino: Path,
) -> None:
    marcador = destino / ".extraido"

    if marcador.exists():
        print("Já extraído:", destino)
        return

    with zipfile.ZipFile(arquivo_zip) as arquivo:
        arquivo.extractall(destino)

    marcador.touch()


if CFG.usar_esc50:
    zip_esc = DADOS / "esc50_master.zip"
    baixar(URL_ESC50, zip_esc)
    extrair_zip(zip_esc, ESC50)

if CFG.usar_tut_rare:
    zip_tut = DADOS / "tut_eventos.zip"
    baixar(URL_TUT_EVENTOS, zip_tut)
    extrair_zip(zip_tut, TUT)

## 3. Montar metadados do ESC-50

In [ ]:
CLASSES_NORMAIS_ESC50 = {
    "rain",
    "sea_waves",
    "crickets",
    "chirping_birds",
    "water_drops",
    "wind",
    "footsteps",
    "laughing",
    "breathing",
    "snoring",
    "clock_tick",
    "washing_machine",
    "vacuum_cleaner",
    "engine",
    "train",
    "airplane",
    "helicopter",
    "fireworks",
    "thunderstorm",
    "siren",
    "clapping",
    "can_opening",
    "keyboard_typing",
    "mouse_click",
}

MAPA_ESC50 = {
    "crying_baby": "choro_distress",
    "glass_breaking": "vidro_quebrando",
    "door_wood_knock": "impacto",
}


def localizar_unico(
    raiz: Path,
    padrao: str,
) -> Path:
    encontrados = list(raiz.rglob(padrao))

    if not encontrados:
        raise FileNotFoundError(
            f"Não encontrei {padrao} em {raiz}"
        )

    return encontrados[0]


registros_esc = []

if CFG.usar_esc50:
    csv_esc = localizar_unico(
        ESC50,
        "esc50.csv",
    )
    audio_esc = localizar_unico(
        ESC50,
        "audio",
    )

    tabela_esc = pd.read_csv(csv_esc)

    for linha in tabela_esc.itertuples():
        categoria = str(linha.category)

        if categoria in MAPA_ESC50:
            classe = MAPA_ESC50[categoria]
        elif categoria in CLASSES_NORMAIS_ESC50:
            classe = "ambiente_normal"
        else:
            continue

        caminho = (
            audio_esc / str(linha.filename)
        ).resolve()

        if not caminho.exists():
            continue

        registros_esc.append(
            {
                "arquivo": str(caminho),
                "classe": classe,
                "fonte": "ESC-50",
                "grupo": (
                    f"esc50:{linha.src_file}"
                ),
                "origem_id": str(linha.src_file),
                "consulta": categoria,
                "licenca": "CC BY-NC",
            }
        )

df_esc = pd.DataFrame(registros_esc)

print("Registros ESC-50:", len(df_esc))

if not df_esc.empty:
    display(
        df_esc.groupby("classe")
        .size()
        .rename("quantidade")
        .to_frame()
    )

## 4. Montar metadados do TUT Rare

In [ ]:
EXTENSOES_AUDIO = {
    ".wav",
    ".flac",
    ".ogg",
    ".mp3",
    ".m4a",
}


def inferir_classe_tut(
    caminho: Path,
) -> str | None:
    texto = (
        str(caminho)
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    if any(
        termo in texto
        for termo in (
            "babycry",
            "baby_cry",
            "babycrying",
            "baby_crying",
        )
    ):
        return "choro_distress"

    if any(
        termo in texto
        for termo in (
            "glassbreak",
            "glass_break",
            "glassbreaking",
            "glass_breaking",
        )
    ):
        return "vidro_quebrando"

    if any(
        termo in texto
        for termo in (
            "gunshot",
            "gun_shot",
            "gunfire",
        )
    ):
        return "tiro"

    return None


registros_tut = []

if CFG.usar_tut_rare:
    arquivos_tut = [
        caminho
        for caminho in TUT.rglob("*")
        if (
            caminho.is_file()
            and caminho.suffix.lower()
            in EXTENSOES_AUDIO
        )
    ]

    for caminho in arquivos_tut:
        classe = inferir_classe_tut(caminho)

        if classe is None:
            continue

        registros_tut.append(
            {
                "arquivo": str(
                    caminho.resolve()
                ),
                "classe": classe,
                "fonte": "TUT Rare 2017",
                "grupo": (
                    f"tut:{caminho.stem}"
                ),
                "origem_id": caminho.stem,
                "consulta": classe,
                "licenca": (
                    "Conforme LICENSE.txt do TUT"
                ),
            }
        )

df_tut = pd.DataFrame(registros_tut)

print("Registros TUT:", len(df_tut))

if not df_tut.empty:
    display(
        df_tut.groupby("classe")
        .size()
        .rename("quantidade")
        .to_frame()
    )

if CFG.usar_tut_rare and df_tut.empty:
    print(
        "Exemplos de caminhos extraídos para diagnóstico:"
    )
    for caminho in list(TUT.rglob("*"))[:40]:
        print(caminho)
    raise RuntimeError(
        "Não foi possível identificar as classes do TUT."
    )

## 5. Baixar classes vocais e tiros pelo Kaggle

Esta versão não depende da API do Freesound.

Datasets usados:

- **ASVP-ESD**:
  - `04` → `choro_distress`
  - `05` → `voz_agressiva`
  - `06` → `grito_panico`
  - `02` → `ambiente_normal`
- **Gunshot Audio Dataset**:
  - todos os áudios → `tiro`

Observação importante:

- `grito_panico` representa o evento acústico;
- palavras como “socorro” pertencem ao ramo `VAD → Whisper → NLP`;
- `voz_agressiva` representa raiva, grunhido e frustração, não garante que exista uma discussão entre duas pessoas.

In [ ]:
SLUG_ASVP = (
    "dejolilandry/"
    "asvpesdspeech-nonspeech-emotional-utterances"
)
SLUG_TIRO = (
    "emrahaydemr/"
    "gunshot-audio-dataset"
)

PASTA_ASVP = Path(
    kagglehub.dataset_download(SLUG_ASVP)
)
PASTA_TIRO_KAGGLE = Path(
    kagglehub.dataset_download(SLUG_TIRO)
)

print("ASVP-ESD:", PASTA_ASVP)
print("Gunshot:", PASTA_TIRO_KAGGLE)

In [ ]:
MAPA_EMOCAO_ASVP = {
    "02": "ambiente_normal",
    "04": "choro_distress",
    "05": "voz_agressiva",
    "06": "grito_panico",
}


def interpretar_nome_asvp(
    caminho: Path,
) -> tuple[str | None, str]:
    partes = caminho.stem.split("-")

    if len(partes) < 3:
        return None, caminho.parent.name

    codigo = partes[2].zfill(2)

    if codigo not in MAPA_EMOCAO_ASVP:
        return None, caminho.parent.name

    # Pela convenção do ASVP-ESD, o ator é o último
    # componente do nome do arquivo.
    ator = (
        partes[-1]
        if len(partes) >= 7
        else caminho.parent.name
    )

    return codigo, ator


registros_asvp = []

arquivos_asvp = [
    caminho
    for caminho in PASTA_ASVP.rglob("*")
    if (
        caminho.is_file()
        and caminho.suffix.lower()
        in EXTENSOES_AUDIO
    )
]

for caminho in arquivos_asvp:
    codigo, ator = interpretar_nome_asvp(caminho)

    if codigo is None:
        continue

    registros_asvp.append(
        {
            "arquivo": str(caminho.resolve()),
            "classe": MAPA_EMOCAO_ASVP[codigo],
            "fonte": "ASVP-ESD Kaggle",
            "grupo": f"asvp_ator:{ator}",
            "origem_id": caminho.stem,
            "consulta": f"emocao_{codigo}",
            "licenca": "Ver licença do dataset ASVP-ESD",
        }
    )

df_asvp = pd.DataFrame(registros_asvp)

print("Arquivos ASVP encontrados:", len(arquivos_asvp))
print("Registros ASVP utilizados:", len(df_asvp))

if df_asvp.empty:
    raise RuntimeError(
        "Não foi possível interpretar os nomes do ASVP-ESD."
    )

display(
    df_asvp.groupby("classe")
    .agg(
        arquivos=("arquivo", "nunique"),
        grupos=("grupo", "nunique"),
    )
)

In [ ]:
registros_tiro_kaggle = []

arquivos_tiro = [
    caminho
    for caminho in PASTA_TIRO_KAGGLE.rglob("*")
    if (
        caminho.is_file()
        and caminho.suffix.lower()
        in EXTENSOES_AUDIO
    )
]

for caminho in arquivos_tiro:
    registros_tiro_kaggle.append(
        {
            "arquivo": str(caminho.resolve()),
            "classe": "tiro",
            "fonte": "Gunshot Kaggle",
            "grupo": (
                "gunshot_kaggle:"
                f"{caminho.parent.name}:"
                f"{caminho.stem}"
            ),
            "origem_id": caminho.stem,
            "consulta": "gunshot",
            "licenca": "Ver licença do dataset no Kaggle",
        }
    )

df_tiro_kaggle = pd.DataFrame(
    registros_tiro_kaggle
)

print(
    "Registros de tiro Kaggle:",
    len(df_tiro_kaggle),
)

if df_tiro_kaggle.empty:
    raise RuntimeError(
        "Nenhum áudio foi encontrado no dataset de tiros."
    )

### Distribuição esperada após o Kaggle

O ASVP-ESD fornece centenas de exemplos de:

- choro;
- raiva, grunhido e frustração;
- grito, medo e pânico;
- voz neutra e calma.

O dataset de tiros adiciona centenas de recortes de disparos.

A auditoria manual continua necessária porque o ASVP-ESD reúne material realista de filmes, televisão, YouTube e outras fontes.

## 6. Unificar e limitar as fontes por classe

In [ ]:
tabelas = [
    tabela
    for tabela in (
        df_esc,
        df_tut,
        df_asvp,
        df_tiro_kaggle,
    )
    if not tabela.empty
]

metadados_brutos = pd.concat(
    tabelas,
    ignore_index=True,
)

metadados_brutos = (
    metadados_brutos[
        metadados_brutos["classe"].isin(CLASSES)
    ]
    .drop_duplicates(subset=["arquivo"])
    .reset_index(drop=True)
)


def limitar_metadados(
    dados: pd.DataFrame,
) -> pd.DataFrame:
    rng = random.Random(CFG.semente)
    partes = []

    for classe in CLASSES:
        parte_classe = dados[
            dados["classe"] == classe
        ].copy()

        grupos = (
            parte_classe["grupo"]
            .drop_duplicates()
            .tolist()
        )

        rng.shuffle(grupos)

        grupos = grupos[
            : CFG.max_grupos_por_classe
        ]

        parte_classe = parte_classe[
            parte_classe["grupo"].isin(grupos)
        ]

        for _, parte_grupo in parte_classe.groupby(
            "grupo",
            sort=False,
        ):
            if (
                len(parte_grupo)
                > CFG.max_arquivos_por_grupo_classe
            ):
                parte_grupo = parte_grupo.sample(
                    CFG.max_arquivos_por_grupo_classe,
                    random_state=CFG.semente,
                )

            partes.append(parte_grupo)

    return (
        pd.concat(partes, ignore_index=True)
        .drop_duplicates(subset=["arquivo"])
        .reset_index(drop=True)
    )


metadados = limitar_metadados(
    metadados_brutos
)

metadados.to_csv(
    RESULTADOS / "fontes_unificadas.csv",
    index=False,
)

resumo_fontes = (
    metadados.groupby(["classe", "fonte"])
    .agg(
        arquivos=("arquivo", "nunique"),
        grupos=("grupo", "nunique"),
    )
)

display(resumo_fontes)

ausentes = [
    classe
    for classe in CLASSES
    if classe not in set(metadados["classe"])
]

if ausentes:
    raise RuntimeError(
        f"Classes sem dados: {ausentes}"
    )

print("Arquivos:", metadados["arquivo"].nunique())
print("Grupos:", metadados["grupo"].nunique())

## 7. Auditoria manual

Ouça exemplos aleatórios. A coleta do Freesound usa metadados e busca textual, então alguns arquivos podem estar rotulados incorretamente.

Remova linhas ruins de `fontes_unificadas.csv` ou marque `APROVAR_AUDITORIA = True` somente depois de conferir.

In [ ]:
AMOSTRAS_POR_CLASSE = 3

for classe in CLASSES:
    print("\n" + "=" * 72)
    print("Classe:", classe)

    parte = metadados[
        metadados["classe"] == classe
    ]

    quantidade = min(
        AMOSTRAS_POR_CLASSE,
        len(parte),
    )

    amostras = parte.sample(
        quantidade,
        random_state=CFG.semente,
    )

    for linha in amostras.itertuples():
        print(
            linha.fonte,
            "|",
            Path(linha.arquivo).name,
            "|",
            getattr(linha, "consulta", ""),
        )
        display(Audio(linha.arquivo))

In [ ]:
APROVAR_AUDITORIA = True

if not APROVAR_AUDITORIA:
    raise RuntimeError(
        "Ouça as amostras e altere "
        "APROVAR_AUDITORIA para True."
    )

## 8. Separação por grupo antes do augmentation

O split é feito por `grupo` dentro de cada classe. Assim, o mesmo áudio original e suas variações nunca atravessam treino, validação e teste.

In [ ]:
SPLITS = [
    "treino",
    "validacao",
    "teste",
]

PROPORCOES_SPLIT = np.array(
    [
        CFG.proporcao_treino,
        CFG.proporcao_validacao,
        CFG.proporcao_teste,
    ],
    dtype=np.float64,
)


def pontuar_divisao(
    dados: pd.DataFrame,
) -> float:
    tabela = (
        dados.groupby(["classe", "split"])["arquivo"]
        .nunique()
        .unstack(fill_value=0)
        .reindex(
            index=CLASSES,
            columns=SPLITS,
            fill_value=0,
        )
    )

    if (tabela == 0).any().any():
        return 1e9

    proporcoes = tabela.div(
        tabela.sum(axis=1),
        axis=0,
    ).to_numpy()

    erro_classes = float(
        np.abs(
            proporcoes
            - PROPORCOES_SPLIT
        ).mean()
    )

    totais = (
        dados.groupby("split")["arquivo"]
        .nunique()
        .reindex(SPLITS, fill_value=0)
        .to_numpy(dtype=np.float64)
    )

    proporcoes_totais = (
        totais / totais.sum()
    )

    erro_total = float(
        np.abs(
            proporcoes_totais
            - PROPORCOES_SPLIT
        ).mean()
    )

    return erro_classes + 0.30 * erro_total


def encontrar_melhor_split(
    dados: pd.DataFrame,
) -> tuple[pd.DataFrame, float]:
    grupos = (
        dados["grupo"]
        .drop_duplicates()
        .to_numpy()
    )

    melhor = None
    melhor_score = float("inf")

    for tentativa in range(
        CFG.tentativas_split
    ):
        rng = np.random.default_rng(
            CFG.semente + tentativa
        )

        atribuicoes = rng.choice(
            SPLITS,
            size=len(grupos),
            p=PROPORCOES_SPLIT,
        )

        mapa = dict(
            zip(
                grupos,
                atribuicoes,
                strict=True,
            )
        )

        candidato = dados.copy()
        candidato["split"] = (
            candidato["grupo"].map(mapa)
        )

        score = pontuar_divisao(
            candidato
        )

        if score < melhor_score:
            melhor = candidato
            melhor_score = score

            if score < 0.025:
                break

    if (
        melhor is None
        or melhor_score >= 1e9
    ):
        raise RuntimeError(
            "Não foi possível criar splits com "
            "todas as classes."
        )

    return melhor, melhor_score


metadados, score_split = encontrar_melhor_split(
    metadados
)

metadados.to_csv(
    RESULTADOS / "fontes_com_split.csv",
    index=False,
)

display(
    metadados.groupby(["classe", "split"])
    .agg(
        arquivos=("arquivo", "nunique"),
        grupos=("grupo", "nunique"),
    )
)

vazamento = (
    metadados.groupby("grupo")["split"]
    .nunique()
    .gt(1)
    .sum()
)

print("Score da divisão:", score_split)
print("Grupos em mais de um split:", vazamento)

if vazamento:
    raise RuntimeError(
        "Foi detectado vazamento entre splits."
    )

## 9. Janelas uniformes, augmentation e features

A função de janelas não recebe o rótulo. Isso elimina a vantagem
artificial que existia no teste anterior.

In [ ]:
AMOSTRAS_JANELA = int(
    CFG.taxa_amostragem
    * CFG.duracao_janela
)

exemplo_zero = np.zeros(
    AMOSTRAS_JANELA,
    dtype=np.float32,
)

mel_zero = librosa.feature.melspectrogram(
    y=exemplo_zero,
    sr=CFG.taxa_amostragem,
    n_fft=CFG.n_fft,
    hop_length=CFG.hop_length,
    n_mels=CFG.n_mels,
    fmin=CFG.fmin,
    fmax=CFG.fmax,
    power=2.0,
)

FRAMES = mel_zero.shape[1]
QUANTIDADE_STATS = 8

print("Log-mel:", (CFG.n_mels, FRAMES, 1))
print("MFCC:", (CFG.n_mfcc, FRAMES, 3))
print("Estatísticas:", (QUANTIDADE_STATS,))


def carregar_audio(
    caminho: str | Path,
) -> np.ndarray:
    audio, _ = librosa.load(
        str(caminho),
        sr=CFG.taxa_amostragem,
        mono=True,
    )

    audio = np.nan_to_num(
        audio.astype(np.float32)
    )

    if len(audio) == 0:
        raise ValueError(
            f"Áudio vazio: {caminho}"
        )

    pico = float(np.max(np.abs(audio)))

    if pico > 1.0:
        audio = audio / pico

    return audio


def ajustar_tamanho(
    audio: np.ndarray,
) -> np.ndarray:
    if len(audio) < AMOSTRAS_JANELA:
        return np.pad(
            audio,
            (
                0,
                AMOSTRAS_JANELA - len(audio),
            ),
        ).astype(np.float32)

    return audio[:AMOSTRAS_JANELA].astype(
        np.float32
    )


def gerar_janelas_uniformes(
    audio: np.ndarray,
    passo_segundos: float,
    max_janelas: int | None,
) -> list[tuple[float, np.ndarray]]:
    # Esta função não recebe a classe.
    if len(audio) <= AMOSTRAS_JANELA:
        return [
            (
                0.0,
                ajustar_tamanho(audio),
            )
        ]

    passo = max(
        1,
        int(
            CFG.taxa_amostragem
            * passo_segundos
        ),
    )

    ultimo_inicio = (
        len(audio) - AMOSTRAS_JANELA
    )

    inicios = list(
        range(
            0,
            ultimo_inicio + 1,
            passo,
        )
    )

    if inicios[-1] != ultimo_inicio:
        inicios.append(ultimo_inicio)

    if (
        max_janelas is not None
        and len(inicios) > max_janelas
    ):
        indices = np.linspace(
            0,
            len(inicios) - 1,
            max_janelas,
            dtype=int,
        )

        inicios = [
            inicios[int(indice)]
            for indice in indices
        ]

    return [
        (
            inicio / CFG.taxa_amostragem,
            ajustar_tamanho(
                audio[
                    inicio:
                    inicio + AMOSTRAS_JANELA
                ]
            ),
        )
        for inicio in inicios
    ]


def deslocar_sem_wrap(
    audio: np.ndarray,
    deslocamento: int,
) -> np.ndarray:
    saida = np.zeros_like(audio)

    if deslocamento > 0:
        saida[deslocamento:] = audio[
            :-deslocamento
        ]
    elif deslocamento < 0:
        saida[:deslocamento] = audio[
            -deslocamento:
        ]
    else:
        saida[:] = audio

    return saida


def augmentar(
    audio: np.ndarray,
    rng: np.random.Generator,
) -> np.ndarray:
    saida = audio.copy()

    ganho_db = rng.uniform(-7.0, 5.0)
    saida *= 10 ** (ganho_db / 20.0)

    deslocamento = int(
        rng.uniform(-0.12, 0.12)
        * len(saida)
    )

    saida = deslocar_sem_wrap(
        saida,
        deslocamento,
    )

    if rng.random() < 0.55:
        snr_db = rng.uniform(
            10.0,
            28.0,
        )

        potencia = float(
            np.mean(saida ** 2)
            + 1e-10
        )

        desvio_ruido = math.sqrt(
            potencia
            / 10 ** (snr_db / 10.0)
        )

        saida += rng.normal(
            0.0,
            desvio_ruido,
            size=saida.shape,
        ).astype(np.float32)

    if rng.random() < 0.20:
        tamanho = int(
            rng.uniform(0.03, 0.12)
            * len(saida)
        )

        inicio = int(
            rng.integers(
                0,
                max(
                    1,
                    len(saida) - tamanho,
                ),
            )
        )

        saida[
            inicio:
            inicio + tamanho
        ] = 0.0

    pico = float(
        np.max(np.abs(saida))
    )

    if pico > 1.0:
        saida = saida / pico

    return saida.astype(np.float32)


def normalizar_matriz(
    matriz: np.ndarray,
) -> np.ndarray:
    media = float(np.mean(matriz))
    desvio = float(np.std(matriz))

    return (
        (matriz - media)
        / (desvio + 1e-6)
    ).astype(np.float32)


def extrair_features(
    audio: np.ndarray,
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
]:
    audio = ajustar_tamanho(audio)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=CFG.taxa_amostragem,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
        n_mels=CFG.n_mels,
        fmin=CFG.fmin,
        fmax=CFG.fmax,
        power=2.0,
    )

    log_mel = librosa.power_to_db(
        mel,
        ref=np.max,
        top_db=80.0,
    )

    mfcc_base = librosa.feature.mfcc(
        S=log_mel,
        n_mfcc=CFG.n_mfcc,
    )

    mfcc_delta = librosa.feature.delta(
        mfcc_base,
        order=1,
    )

    mfcc_delta2 = librosa.feature.delta(
        mfcc_base,
        order=2,
    )

    mfcc = np.stack(
        [
            normalizar_matriz(mfcc_base),
            normalizar_matriz(mfcc_delta),
            normalizar_matriz(mfcc_delta2),
        ],
        axis=-1,
    ).astype(np.float32)

    log_mel = normalizar_matriz(
        log_mel
    )[..., np.newaxis]

    rms = librosa.feature.rms(
        y=audio,
        frame_length=CFG.n_fft,
        hop_length=CFG.hop_length,
    )[0]

    zcr = librosa.feature.zero_crossing_rate(
        audio,
        frame_length=CFG.n_fft,
        hop_length=CFG.hop_length,
    )[0]

    centroide = librosa.feature.spectral_centroid(
        y=audio,
        sr=CFG.taxa_amostragem,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
    )[0]

    largura = librosa.feature.spectral_bandwidth(
        y=audio,
        sr=CFG.taxa_amostragem,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
    )[0]

    rolloff = librosa.feature.spectral_rolloff(
        y=audio,
        sr=CFG.taxa_amostragem,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
        roll_percent=0.85,
    )[0]

    flatness = librosa.feature.spectral_flatness(
        y=audio,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
    )[0]

    stats = np.array(
        [
            np.mean(rms),
            np.std(rms),
            np.mean(zcr),
            np.std(zcr),
            np.mean(centroide)
            / (CFG.taxa_amostragem / 2),
            np.mean(largura)
            / (CFG.taxa_amostragem / 2),
            np.mean(rolloff)
            / (CFG.taxa_amostragem / 2),
            np.mean(flatness),
        ],
        dtype=np.float32,
    )

    return (
        log_mel.astype(np.float32),
        mfcc,
        stats,
    )

In [ ]:
def processar_split(
    nome_split: str,
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    pd.DataFrame,
]:
    parte = metadados[
        metadados["split"] == nome_split
    ].copy()

    mel_lista = []
    mfcc_lista = []
    stats_lista = []
    rotulos = []
    registros = []
    erros = []

    rng = np.random.default_rng(
        CFG.semente
        + {
            "treino": 1,
            "validacao": 2,
            "teste": 3,
        }[nome_split]
    )

    for linha in tqdm(
        parte.itertuples(),
        total=len(parte),
        desc=f"Processando {nome_split}",
    ):
        try:
            audio = carregar_audio(
                linha.arquivo
            )

            janelas = gerar_janelas_uniformes(
                audio,
                CFG.passo_janela_dataset,
                CFG.max_janelas_por_fonte,
            )
        except Exception as erro:
            erros.append(
                {
                    "arquivo": linha.arquivo,
                    "erro": str(erro),
                }
            )
            continue

        for (
            indice_janela,
            (
                inicio_segundos,
                janela,
            ),
        ) in enumerate(janelas):
            versoes = [
                ("original", janela)
            ]

            if nome_split == "treino":
                for copia in range(
                    CFG.copias_augmentation_treino
                ):
                    versoes.append(
                        (
                            f"aug_{copia + 1}",
                            augmentar(
                                janela,
                                rng,
                            ),
                        )
                    )

            for versao, audio_versao in versoes:
                (
                    log_mel,
                    mfcc,
                    stats,
                ) = extrair_features(
                    audio_versao
                )

                mel_lista.append(log_mel)
                mfcc_lista.append(mfcc)
                stats_lista.append(stats)
                rotulos.append(
                    ID_CLASSE[linha.classe]
                )

                registros.append(
                    {
                        "split": nome_split,
                        "classe": linha.classe,
                        "fonte": linha.fonte,
                        "grupo": linha.grupo,
                        "arquivo": linha.arquivo,
                        "janela": indice_janela,
                        "inicio_segundos": (
                            inicio_segundos
                        ),
                        "fim_segundos": (
                            inicio_segundos
                            + CFG.duracao_janela
                        ),
                        "versao": versao,
                    }
                )

    if erros:
        pd.DataFrame(erros).to_csv(
            RESULTADOS
            / f"erros_{nome_split}.csv",
            index=False,
        )

    if not mel_lista:
        raise RuntimeError(
            f"Nenhuma janela criada para {nome_split}."
        )

    return (
        np.asarray(
            mel_lista,
            dtype=np.float32,
        ),
        np.asarray(
            mfcc_lista,
            dtype=np.float32,
        ),
        np.asarray(
            stats_lista,
            dtype=np.float32,
        ),
        np.asarray(
            rotulos,
            dtype=np.int64,
        ),
        pd.DataFrame(registros),
    )


ASSINATURA_CACHE = hashlib.sha1(
    json.dumps(
        {
            "configuracao": asdict(CFG),
            "classes": CLASSES,
            "versao": (
                "v2_janelas_uniformes_"
                "mfcc_delta_stats"
            ),
        },
        sort_keys=True,
        ensure_ascii=False,
    ).encode("utf-8")
).hexdigest()

caminho_assinatura = (
    PROCESSADO
    / "assinatura_cache.txt"
)

assinatura_anterior = (
    caminho_assinatura.read_text()
    if caminho_assinatura.exists()
    else ""
)

if (
    CFG.reprocessar_features
    or assinatura_anterior
    != ASSINATURA_CACHE
):
    for caminho in PROCESSADO.iterdir():
        if caminho.is_file():
            caminho.unlink()
        elif caminho.is_dir():
            shutil.rmtree(caminho)

    caminho_assinatura.write_text(
        ASSINATURA_CACHE,
        encoding="utf-8",
    )


dados_processados = {}

for split in (
    "treino",
    "validacao",
    "teste",
):
    cache = PROCESSADO / f"{split}.npz"
    cache_meta = PROCESSADO / f"{split}.csv"

    if (
        cache.exists()
        and cache_meta.exists()
        and not CFG.reprocessar_features
    ):
        arquivo = np.load(cache)

        dados_processados[split] = (
            arquivo["mel"],
            arquivo["mfcc"],
            arquivo["stats"],
            arquivo["y"],
            pd.read_csv(cache_meta),
        )

        print("Cache carregado:", split)
        continue

    (
        mel,
        mfcc,
        stats,
        y,
        meta,
    ) = processar_split(split)

    np.savez_compressed(
        cache,
        mel=mel,
        mfcc=mfcc,
        stats=stats,
        y=y,
    )

    meta.to_csv(
        cache_meta,
        index=False,
    )

    dados_processados[split] = (
        mel,
        mfcc,
        stats,
        y,
        meta,
    )

for split, (
    mel,
    mfcc,
    stats,
    y,
    meta,
) in dados_processados.items():
    print(
        split,
        "mel=",
        mel.shape,
        "mfcc=",
        mfcc.shape,
        "stats=",
        stats.shape,
        "y=",
        y.shape,
    )

## 10. Visualizar log-mel e MFCC

In [ ]:
(
    X_mel_treino,
    X_mfcc_treino,
    X_stats_treino,
    y_treino,
    meta_treino,
) = dados_processados["treino"]

(
    X_mel_val,
    X_mfcc_val,
    X_stats_val,
    y_val,
    meta_val,
) = dados_processados["validacao"]

(
    X_mel_teste,
    X_mfcc_teste,
    X_stats_teste,
    y_teste,
    meta_teste,
) = dados_processados["teste"]


MEDIA_STATS = X_stats_treino.mean(
    axis=0
).astype(np.float32)

DESVIO_STATS = X_stats_treino.std(
    axis=0
).astype(np.float32)

DESVIO_STATS = np.where(
    DESVIO_STATS < 1e-6,
    1.0,
    DESVIO_STATS,
).astype(np.float32)


def normalizar_stats_lote(
    valores: np.ndarray,
) -> np.ndarray:
    return (
        (
            valores
            - MEDIA_STATS
        )
        / DESVIO_STATS
    ).astype(np.float32)


X_stats_treino = normalizar_stats_lote(
    X_stats_treino
)

X_stats_val = normalizar_stats_lote(
    X_stats_val
)

X_stats_teste = normalizar_stats_lote(
    X_stats_teste
)

normalizacao_stats = {
    "media": MEDIA_STATS.tolist(),
    "desvio": DESVIO_STATS.tolist(),
}

(
    MODELOS / "normalizacao_stats.json"
).write_text(
    json.dumps(
        normalizacao_stats,
        indent=2,
    ),
    encoding="utf-8",
)

display(
    pd.DataFrame(
        {
            "treino": Counter(
                CLASSES[int(valor)]
                for valor in y_treino
            ),
            "validacao": Counter(
                CLASSES[int(valor)]
                for valor in y_val
            ),
            "teste": Counter(
                CLASSES[int(valor)]
                for valor in y_teste
            ),
        }
    ).fillna(0).astype(int)
)

indice = 0

plt.figure(figsize=(12, 4))
plt.imshow(
    X_mel_treino[indice, :, :, 0],
    aspect="auto",
    origin="lower",
)
plt.title(
    "Log-mel — "
    + CLASSES[int(y_treino[indice])]
)
plt.xlabel("Tempo")
plt.ylabel("Bandas mel")
plt.colorbar()
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.imshow(
    X_mfcc_treino[indice, :, :, 0],
    aspect="auto",
    origin="lower",
)
plt.title(
    "MFCC — "
    + CLASSES[int(y_treino[indice])]
)
plt.xlabel("Tempo")
plt.ylabel("Coeficientes MFCC")
plt.colorbar()
plt.tight_layout()
plt.show()

## 11. Criar a CNN leve de três entradas

O modelo combina log-mel, MFCC com delta/delta² e estatísticas acústicas.

In [ ]:
def bloco_separavel(
    entrada: tf.Tensor,
    filtros: int,
    reduzir: bool = True,
) -> tf.Tensor:
    atalho = entrada

    x = tf.keras.layers.SeparableConv2D(
        filtros,
        3,
        padding="same",
        use_bias=False,
    )(entrada)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.SeparableConv2D(
        filtros,
        3,
        padding="same",
        use_bias=False,
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)

    if atalho.shape[-1] != filtros:
        atalho = tf.keras.layers.Conv2D(
            filtros,
            1,
            padding="same",
            use_bias=False,
        )(atalho)

        atalho = tf.keras.layers.BatchNormalization()(
            atalho
        )

    x = tf.keras.layers.Add()(
        [x, atalho]
    )

    x = tf.keras.layers.ReLU()(x)

    if reduzir:
        x = tf.keras.layers.MaxPooling2D(
            2
        )(x)

    return x


entrada_mel = tf.keras.Input(
    shape=(CFG.n_mels, FRAMES, 1),
    name="log_mel",
)

entrada_mfcc = tf.keras.Input(
    shape=(CFG.n_mfcc, FRAMES, 3),
    name="mfcc",
)

entrada_stats = tf.keras.Input(
    shape=(QUANTIDADE_STATS,),
    name="estatisticas",
)


ramo_mel = bloco_separavel(
    entrada_mel,
    16,
)

ramo_mel = bloco_separavel(
    ramo_mel,
    32,
)

ramo_mel = bloco_separavel(
    ramo_mel,
    64,
    reduzir=False,
)

ramo_mel = tf.keras.layers.GlobalAveragePooling2D()(
    ramo_mel
)


ramo_mfcc = bloco_separavel(
    entrada_mfcc,
    12,
)

ramo_mfcc = bloco_separavel(
    ramo_mfcc,
    24,
)

ramo_mfcc = bloco_separavel(
    ramo_mfcc,
    48,
    reduzir=False,
)

ramo_mfcc = tf.keras.layers.GlobalAveragePooling2D()(
    ramo_mfcc
)


ramo_stats = tf.keras.layers.Dense(
    24,
    activation="relu",
)(entrada_stats)

ramo_stats = tf.keras.layers.BatchNormalization()(
    ramo_stats
)


x = tf.keras.layers.Concatenate()(
    [
        ramo_mel,
        ramo_mfcc,
        ramo_stats,
    ]
)

x = tf.keras.layers.Dense(
    96,
    activation="relu",
    kernel_regularizer=(
        tf.keras.regularizers.l2(
            1e-4
        )
    ),
)(x)

x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.40)(x)

saida = tf.keras.layers.Dense(
    len(CLASSES),
    activation="softmax",
    name="probabilidades",
)(x)

modelo = tf.keras.Model(
    inputs={
        "log_mel": entrada_mel,
        "mfcc": entrada_mfcc,
        "estatisticas": entrada_stats,
    },
    outputs=saida,
    name="aed_multiclasse_v2",
)

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(
        CFG.taxa_aprendizado
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=2,
            name="top2_accuracy",
        ),
    ],
)

modelo.summary()

## 12. Treinar

In [ ]:
pesos = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASSES)),
    y=y_treino,
)

pesos_classes = {
    indice: float(peso)
    for indice, peso in enumerate(pesos)
}

print(
    "Pesos:",
    {
        CLASSES[indice]: peso
        for indice, peso
        in pesos_classes.items()
    },
)


class MacroF1Callback(
    tf.keras.callbacks.Callback
):
    def __init__(
        self,
        mel: np.ndarray,
        mfcc: np.ndarray,
        stats: np.ndarray,
        y: np.ndarray,
    ) -> None:
        super().__init__()

        self.mel = mel
        self.mfcc = mfcc
        self.stats = stats
        self.y = y
        self.valores = []

    def on_epoch_end(
        self,
        epoch: int,
        logs: dict | None = None,
    ) -> None:
        logs = logs or {}

        probabilidades = self.model.predict(
            {
                "log_mel": self.mel,
                "mfcc": self.mfcc,
                "estatisticas": self.stats,
            },
            batch_size=CFG.tamanho_lote,
            verbose=0,
        )

        previsoes = np.argmax(
            probabilidades,
            axis=1,
        )

        valor = float(
            f1_score(
                self.y,
                previsoes,
                average="macro",
                zero_division=0,
            )
        )

        logs["val_macro_f1"] = valor
        self.valores.append(valor)

        print(
            f" — val_macro_f1: {valor:.4f}"
        )


callback_f1 = MacroF1Callback(
    X_mel_val,
    X_mfcc_val,
    X_stats_val,
    y_val,
)

caminho_melhor = (
    MODELOS / "aed_v2_melhor.keras"
)

callbacks = [
    callback_f1,
    tf.keras.callbacks.ModelCheckpoint(
        caminho_melhor,
        monitor="val_macro_f1",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_macro_f1",
        mode="max",
        patience=CFG.paciencia,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(
        RESULTADOS
        / "historico_treinamento.csv"
    ),
]

historico = modelo.fit(
    {
        "log_mel": X_mel_treino,
        "mfcc": X_mfcc_treino,
        "estatisticas": X_stats_treino,
    },
    y_treino,
    validation_data=(
        {
            "log_mel": X_mel_val,
            "mfcc": X_mfcc_val,
            "estatisticas": X_stats_val,
        },
        y_val,
    ),
    epochs=CFG.epocas,
    batch_size=CFG.tamanho_lote,
    class_weight=pesos_classes,
    callbacks=callbacks,
    verbose=1,
)

modelo = tf.keras.models.load_model(
    caminho_melhor
)

In [ ]:
historico_df = pd.DataFrame(
    historico.history
)

if (
    "val_macro_f1"
    not in historico_df.columns
):
    historico_df[
        "val_macro_f1"
    ] = callback_f1.valores

historico_df.to_csv(
    RESULTADOS
    / "historico_treinamento_completo.csv",
    index=False,
)

plt.figure(figsize=(10, 5))
plt.plot(
    historico_df["loss"],
    label="treino",
)
plt.plot(
    historico_df["val_loss"],
    label="validação",
)
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(
    historico_df["accuracy"],
    label="acurácia treino",
)
plt.plot(
    historico_df["val_accuracy"],
    label="acurácia validação",
)
plt.plot(
    historico_df["val_macro_f1"],
    label="F1 macro validação",
)
plt.xlabel("Época")
plt.ylabel("Métrica")
plt.title("Acurácia e F1 macro")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

## 13. Gerar probabilidades de validação

In [ ]:
probabilidades_val = modelo.predict(
    {
        "log_mel": X_mel_val,
        "mfcc": X_mfcc_val,
        "estatisticas": X_stats_val,
    },
    batch_size=CFG.tamanho_lote,
    verbose=1,
)

print(
    "Probabilidades de validação:",
    probabilidades_val.shape,
)

## 14. Calibrar limiares na validação

Os limiares são definidos usando apenas a validação. O teste permanece intocado.

In [ ]:
def calibrar_limiar_classe(
    y_real: np.ndarray,
    probabilidades: np.ndarray,
    indice_classe: int,
) -> tuple[float, float, float, float]:
    y_binario = (
        y_real == indice_classe
    ).astype(int)

    precisao, recall, limiares_pr = (
        precision_recall_curve(
            y_binario,
            probabilidades[:, indice_classe],
        )
    )

    if len(limiares_pr) == 0:
        return 0.50, 0.0, 0.0, 0.0

    precisao = precisao[:-1]
    recall = recall[:-1]

    f1 = (
        2
        * precisao
        * recall
        / (
            precisao
            + recall
            + 1e-12
        )
    )

    mascara_intervalo = (
        (limiares_pr >= CFG.limiar_minimo)
        & (
            limiares_pr
            <= CFG.limiar_maximo
        )
    )

    mascara_precisao = (
        precisao
        >= CFG.precisao_minima_limiar
    )

    candidatos = np.where(
        mascara_intervalo
        & mascara_precisao
    )[0]

    if len(candidatos) == 0:
        candidatos = np.where(
            mascara_intervalo
        )[0]

    if len(candidatos) == 0:
        candidatos = np.arange(
            len(limiares_pr)
        )

    melhor = candidatos[
        int(
            np.argmax(
                f1[candidatos]
            )
        )
    ]

    return (
        float(limiares_pr[melhor]),
        float(f1[melhor]),
        float(precisao[melhor]),
        float(recall[melhor]),
    )


limiares = {
    "ambiente_normal": 0.0
}

registros_limiares = []

for classe in CLASSES_EVENTO:
    indice = ID_CLASSE[classe]

    (
        limiar,
        f1_val,
        precisao_val,
        recall_val,
    ) = calibrar_limiar_classe(
        y_val,
        probabilidades_val,
        indice,
    )

    limiares[classe] = round(
        limiar,
        4,
    )

    registros_limiares.append(
        {
            "classe": classe,
            "limiar": limiar,
            "f1_validacao": f1_val,
            "precisao_validacao": (
                precisao_val
            ),
            "recall_validacao": (
                recall_val
            ),
        }
    )

tabela_limiares = pd.DataFrame(
    registros_limiares
)

display(tabela_limiares)

tabela_limiares.to_csv(
    RESULTADOS
    / "limiares_por_classe.csv",
    index=False,
)

(
    MODELOS / "limiares.json"
).write_text(
    json.dumps(
        limiares,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


def decidir_classes(
    probabilidades: np.ndarray,
) -> np.ndarray:
    resultado = []

    for vetor in probabilidades:
        indice = int(np.argmax(vetor))
        classe = CLASSES[indice]

        if classe == "ambiente_normal":
            resultado.append(
                ID_CLASSE[
                    "ambiente_normal"
                ]
            )
            continue

        if (
            float(vetor[indice])
            >= limiares[classe]
        ):
            resultado.append(indice)
        else:
            resultado.append(
                ID_CLASSE[
                    "ambiente_normal"
                ]
            )

    return np.asarray(
        resultado,
        dtype=np.int64,
    )


probabilidades_teste = modelo.predict(
    {
        "log_mel": X_mel_teste,
        "mfcc": X_mfcc_teste,
        "estatisticas": X_stats_teste,
    },
    batch_size=CFG.tamanho_lote,
    verbose=1,
)

previsoes_teste = decidir_classes(
    probabilidades_teste
)

metricas_janela = {
    "acuracia": float(
        accuracy_score(
            y_teste,
            previsoes_teste,
        )
    ),
    "acuracia_balanceada": float(
        balanced_accuracy_score(
            y_teste,
            previsoes_teste,
        )
    ),
    "f1_macro": float(
        f1_score(
            y_teste,
            previsoes_teste,
            average="macro",
            zero_division=0,
        )
    ),
    "f1_ponderado": float(
        f1_score(
            y_teste,
            previsoes_teste,
            average="weighted",
            zero_division=0,
        )
    ),
}

print(
    json.dumps(
        metricas_janela,
        indent=2,
        ensure_ascii=False,
    )
)

relatorio_dict = classification_report(
    y_teste,
    previsoes_teste,
    labels=np.arange(len(CLASSES)),
    target_names=CLASSES,
    output_dict=True,
    zero_division=0,
)

display(
    pd.DataFrame(
        relatorio_dict
    ).T
)

pd.DataFrame(
    relatorio_dict
).T.to_csv(
    RESULTADOS
    / "metricas_teste_por_classe.csv"
)

matriz = confusion_matrix(
    y_teste,
    previsoes_teste,
    labels=np.arange(len(CLASSES)),
)

figura, eixo = plt.subplots(
    figsize=(11, 9)
)

ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=CLASSES,
).plot(
    ax=eixo,
    xticks_rotation=45,
    values_format="d",
    colorbar=False,
)

eixo.set_title(
    "Matriz de confusão — janelas"
)

figura.tight_layout()
figura.savefig(
    RESULTADOS
    / "matriz_confusao_teste.png",
    dpi=170,
)
plt.show()

predicoes_df = meta_teste.copy()
predicoes_df["classe_real"] = [
    CLASSES[int(indice)]
    for indice in y_teste
]
predicoes_df["classe_prevista"] = [
    CLASSES[int(indice)]
    for indice in previsoes_teste
]
predicoes_df["confianca"] = np.max(
    probabilidades_teste,
    axis=1,
)

for indice, classe in enumerate(CLASSES):
    predicoes_df[
        f"prob_{classe}"
    ] = probabilidades_teste[
        :,
        indice,
    ]

predicoes_df.to_csv(
    RESULTADOS
    / "predicoes_teste.csv",
    index=False,
)

## 15. Avaliação por gravação

In [ ]:
def agregar_por_gravacao(
    probabilidades: np.ndarray,
    y_real: np.ndarray,
    metadados_janelas: pd.DataFrame,
    top_k: int = 3,
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    pd.DataFrame,
]:
    tabela = (
        metadados_janelas
        .reset_index(drop=True)
        .copy()
    )

    tabela["indice_array"] = np.arange(
        len(tabela)
    )

    vetores = []
    reais = []
    linhas = []

    for arquivo, parte in tabela.groupby(
        "arquivo",
        sort=False,
    ):
        indices = parte[
            "indice_array"
        ].to_numpy()

        probs = probabilidades[
            indices
        ]

        agregado = []

        for indice_classe in range(
            len(CLASSES)
        ):
            valores = np.sort(
                probs[:, indice_classe]
            )

            quantidade = min(
                top_k,
                len(valores),
            )

            agregado.append(
                float(
                    valores[
                        -quantidade:
                    ].mean()
                )
            )

        vetor = np.asarray(
            agregado,
            dtype=np.float32,
        )

        vetor = vetor / (
            vetor.sum() + 1e-12
        )

        vetores.append(vetor)

        real = int(
            y_real[
                indices[0]
            ]
        )

        reais.append(real)

        linhas.append(
            {
                "arquivo": arquivo,
                "grupo": (
                    parte["grupo"].iloc[0]
                ),
                "fonte": (
                    parte["fonte"].iloc[0]
                ),
                "classe_real": (
                    CLASSES[real]
                ),
                "quantidade_janelas": (
                    len(indices)
                ),
            }
        )

    probabilidades_agregadas = (
        np.asarray(
            vetores,
            dtype=np.float32,
        )
    )

    reais_array = np.asarray(
        reais,
        dtype=np.int64,
    )

    previstos = decidir_classes(
        probabilidades_agregadas
    )

    resumo = pd.DataFrame(linhas)

    resumo["classe_prevista"] = [
        CLASSES[int(indice)]
        for indice in previstos
    ]

    resumo["confianca"] = np.max(
        probabilidades_agregadas,
        axis=1,
    )

    return (
        probabilidades_agregadas,
        reais_array,
        previstos,
        resumo,
    )


(
    probabilidades_gravacao,
    y_gravacao,
    previsoes_gravacao,
    predicoes_gravacao_df,
) = agregar_por_gravacao(
    probabilidades_teste,
    y_teste,
    meta_teste,
)

metricas_gravacao = {
    "acuracia": float(
        accuracy_score(
            y_gravacao,
            previsoes_gravacao,
        )
    ),
    "acuracia_balanceada": float(
        balanced_accuracy_score(
            y_gravacao,
            previsoes_gravacao,
        )
    ),
    "f1_macro": float(
        f1_score(
            y_gravacao,
            previsoes_gravacao,
            average="macro",
            zero_division=0,
        )
    ),
    "f1_ponderado": float(
        f1_score(
            y_gravacao,
            previsoes_gravacao,
            average="weighted",
            zero_division=0,
        )
    ),
}

print(
    json.dumps(
        metricas_gravacao,
        indent=2,
        ensure_ascii=False,
    )
)

relatorio_gravacao = classification_report(
    y_gravacao,
    previsoes_gravacao,
    labels=np.arange(len(CLASSES)),
    target_names=CLASSES,
    output_dict=True,
    zero_division=0,
)

display(
    pd.DataFrame(
        relatorio_gravacao
    ).T
)

pd.DataFrame(
    relatorio_gravacao
).T.to_csv(
    RESULTADOS
    / "metricas_gravacao_por_classe.csv"
)

predicoes_gravacao_df.to_csv(
    RESULTADOS
    / "predicoes_teste_gravacao.csv",
    index=False,
)

matriz_gravacao = confusion_matrix(
    y_gravacao,
    previsoes_gravacao,
    labels=np.arange(len(CLASSES)),
)

figura, eixo = plt.subplots(
    figsize=(11, 9)
)

ConfusionMatrixDisplay(
    confusion_matrix=matriz_gravacao,
    display_labels=CLASSES,
).plot(
    ax=eixo,
    xticks_rotation=45,
    values_format="d",
    colorbar=False,
)

eixo.set_title(
    "Matriz de confusão por gravação"
)

figura.tight_layout()

figura.savefig(
    RESULTADOS
    / "matriz_confusao_gravacao.png",
    dpi=170,
)

plt.show()

## 17. Exportar Keras e TFLite

In [ ]:
caminho_final_keras = (
    MODELOS / "aed_multiclasse_v2_final.keras"
)

modelo.save(caminho_final_keras)

# Save the Keras model in SavedModel format first
caminho_saved_model = MODELOS / "aed_multiclasse_v2_saved_model"
tf.saved_model.save(modelo, str(caminho_saved_model))

# Convert to TFLite Float32 using from_saved_model
converter = (
    tf.lite.TFLiteConverter
    .from_saved_model(str(caminho_saved_model))
)

tflite_float32 = converter.convert()

caminho_float32 = (
    MODELOS / "aed_v2_float32.tflite"
)

caminho_float32.write_bytes(
    tflite_float32
)

# Convert to TFLite Float16 using from_saved_model
converter = (
    tf.lite.TFLiteConverter
    .from_saved_model(str(caminho_saved_model))
)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

converter.target_spec.supported_types = [
    tf.float16
]

tflite_float16 = converter.convert()

caminho_float16 = (
    MODELOS / "aed_v2_float16.tflite"
)

caminho_float16.write_bytes(
    tflite_float16
)


def dados_representativos():
    limite = min(
        400,
        len(X_mel_treino),
    )

    indices = np.linspace(
        0,
        len(X_mel_treino) - 1,
        limite,
        dtype=int,
    )

    for indice in indices:
        yield {
            "log_mel": (
                X_mel_treino[
                    indice:
                    indice + 1
                ]
            ),
            "mfcc": (
                X_mfcc_treino[
                    indice:
                    indice + 1
                ]
            ),
            "estatisticas": (
                X_stats_treino[
                    indice:
                    indice + 1
                ]
            ),
        }


caminho_int8 = (
    MODELOS / "aed_v2_int8.tflite"
)

try:
    # Convert to TFLite INT8 using from_saved_model
    converter = (
        tf.lite.TFLiteConverter
        .from_saved_model(str(caminho_saved_model))
    )

    converter.optimizations = [
        tf.lite.Optimize.DEFAULT
    ]

    converter.representative_dataset = (
        dados_representativos
    )

    converter.target_spec.supported_ops = [
        tf.lite.OpsSet
        .TFLITE_BUILTINS_INT8
    ]

    converter.inference_input_type = (
        tf.int8
    )

    converter.inference_output_type = (
        tf.int8
    )

    tflite_int8 = converter.convert()

    caminho_int8.write_bytes(
        tflite_int8
    )

    print("INT8 exportado.")

except Exception as erro:
    print(
        "Exportação INT8 falhou:",
        erro,
    )

for caminho in (
    caminho_final_keras,
    caminho_float32,
    caminho_float16,
    caminho_int8,
):
    if caminho.exists():
        print(
            caminho.name,
            f"{caminho.stat().st_size / 1e6:.2f} MB",
        )


## 18. Conferir paridade Keras × TFLite

In [ ]:
def identificar_entrada(
    nome: str,
    formato: np.ndarray,
) -> str:
    nome = nome.lower()

    if "log_mel" in nome:
        return "log_mel"

    if "mfcc" in nome:
        return "mfcc"

    if "estat" in nome:
        return "estatisticas"

    if len(formato) == 4:
        if formato[1] == CFG.n_mels:
            return "log_mel"

        return "mfcc"

    return "estatisticas"


def quantizar_tensor(
    valor: np.ndarray,
    detalhe: dict,
) -> np.ndarray:
    dtype = detalhe["dtype"]

    if dtype not in (
        np.int8,
        np.uint8,
    ):
        return valor.astype(dtype)

    escala, zero = detalhe[
        "quantization"
    ]

    if escala == 0:
        return valor.astype(dtype)

    quantizado = np.round(
        valor / escala + zero
    )

    limites = np.iinfo(dtype)

    return np.clip(
        quantizado,
        limites.min,
        limites.max,
    ).astype(dtype)


def desquantizar_tensor(
    valor: np.ndarray,
    detalhe: dict,
) -> np.ndarray:
    if detalhe["dtype"] not in (
        np.int8,
        np.uint8,
    ):
        return valor.astype(np.float32)

    escala, zero = detalhe[
        "quantization"
    ]

    if escala == 0:
        return valor.astype(np.float32)

    return (
        (
            valor.astype(np.float32)
            - zero
        )
        * escala
    )


def prever_tflite_lote(
    caminho: Path,
    mel: np.ndarray,
    mfcc: np.ndarray,
    stats: np.ndarray,
) -> np.ndarray:
    interpretador = tf.lite.Interpreter(
        model_path=str(caminho)
    )

    interpretador.allocate_tensors()

    entradas = (
        interpretador.get_input_details()
    )

    saida = (
        interpretador.get_output_details()[0]
    )

    resultados = []

    for indice in tqdm(
        range(len(mel)),
        desc=caminho.name,
    ):
        valores = {
            "log_mel": mel[
                indice:
                indice + 1
            ],
            "mfcc": mfcc[
                indice:
                indice + 1
            ],
            "estatisticas": stats[
                indice:
                indice + 1
            ],
        }

        for detalhe in entradas:
            chave = identificar_entrada(
                detalhe["name"],
                detalhe["shape"],
            )

            interpretador.set_tensor(
                detalhe["index"],
                quantizar_tensor(
                    valores[chave],
                    detalhe,
                ),
            )

        interpretador.invoke()

        bruto = interpretador.get_tensor(
            saida["index"]
        )

        resultados.append(
            desquantizar_tensor(
                bruto,
                saida,
            )[0]
        )

    return np.asarray(
        resultados,
        dtype=np.float32,
    )


limite_paridade = min(
    250,
    len(y_teste),
)

indices_paridade = np.linspace(
    0,
    len(y_teste) - 1,
    limite_paridade,
    dtype=int,
)

mel_paridade = X_mel_teste[
    indices_paridade
]

mfcc_paridade = X_mfcc_teste[
    indices_paridade
]

stats_paridade = X_stats_teste[
    indices_paridade
]

y_paridade = y_teste[
    indices_paridade
]

pred_keras = modelo.predict(
    {
        "log_mel": mel_paridade,
        "mfcc": mfcc_paridade,
        "estatisticas": stats_paridade,
    },
    verbose=0,
)

registros_paridade = []

for nome, caminho in (
    ("float32", caminho_float32),
    ("float16", caminho_float16),
    ("int8", caminho_int8),
):
    if not caminho.exists():
        continue

    pred_tflite = prever_tflite_lote(
        caminho,
        mel_paridade,
        mfcc_paridade,
        stats_paridade,
    )

    classe_keras = np.argmax(
        pred_keras,
        axis=1,
    )

    classe_tflite = np.argmax(
        pred_tflite,
        axis=1,
    )

    registros_paridade.append(
        {
            "modelo": nome,
            "diferenca_maxima": float(
                np.max(
                    np.abs(
                        pred_keras
                        - pred_tflite
                    )
                )
            ),
            "diferenca_media": float(
                np.mean(
                    np.abs(
                        pred_keras
                        - pred_tflite
                    )
                )
            ),
            "concordancia_argmax": float(
                np.mean(
                    classe_keras
                    == classe_tflite
                )
            ),
            "acuracia_tflite": float(
                accuracy_score(
                    y_paridade,
                    classe_tflite,
                )
            ),
        }
    )

tabela_paridade = pd.DataFrame(
    registros_paridade
)

display(tabela_paridade)

tabela_paridade.to_csv(
    RESULTADOS
    / "paridade_tflite.csv",
    index=False,
)

## 19. Inferência contínua realista

A inferência usa janelas de 2 segundos, passo de 0,5 segundo,
suavização temporal, limiares calibrados e confirmação por
janelas consecutivas.

In [ ]:
def normalizar_stats_unico(
    stats: np.ndarray,
) -> np.ndarray:
    return (
        (
            stats
            - MEDIA_STATS
        )
        / DESVIO_STATS
    ).astype(np.float32)


def aplicar_suavizacao(
    probabilidades: np.ndarray,
) -> np.ndarray:
    if CFG.suavizacao_janelas <= 1:
        return probabilidades

    return (
        pd.DataFrame(probabilidades)
        .rolling(
            window=CFG.suavizacao_janelas,
            min_periods=1,
        )
        .mean()
        .to_numpy(dtype=np.float32)
    )


def inferir_audio_continuo(
    caminho: Path,
) -> pd.DataFrame:
    audio = carregar_audio(caminho)

    janelas = gerar_janelas_uniformes(
        audio,
        CFG.passo_janela_inferencia,
        None,
    )

    mel_lista = []
    mfcc_lista = []
    stats_lista = []
    inicios = []

    for inicio, janela in janelas:
        (
            log_mel,
            mfcc,
            stats,
        ) = extrair_features(janela)

        mel_lista.append(log_mel)
        mfcc_lista.append(mfcc)
        stats_lista.append(
            normalizar_stats_unico(stats)
        )
        inicios.append(inicio)

    probs = modelo.predict(
        {
            "log_mel": np.asarray(
                mel_lista,
                dtype=np.float32,
            ),
            "mfcc": np.asarray(
                mfcc_lista,
                dtype=np.float32,
            ),
            "estatisticas": np.asarray(
                stats_lista,
                dtype=np.float32,
            ),
        },
        batch_size=CFG.tamanho_lote,
        verbose=0,
    )

    probs = aplicar_suavizacao(
        probs
    )

    previsoes = decidir_classes(
        probs
    )

    linhas = []
    consecutivas = 0
    classe_anterior = None

    for (
        inicio,
        vetor,
        indice_previsto,
    ) in zip(
        inicios,
        probs,
        previsoes,
        strict=True,
    ):
        classe = CLASSES[
            int(indice_previsto)
        ]

        evento_ativo = (
            classe != "ambiente_normal"
        )

        if (
            evento_ativo
            and classe == classe_anterior
        ):
            consecutivas += 1
        elif evento_ativo:
            consecutivas = 1
        else:
            consecutivas = 0

        alerta_inicio = (
            evento_ativo
            and consecutivas
            == CFG.minimo_eventos_consecutivos
        )

        alerta_ativo = (
            evento_ativo
            and consecutivas
            >= CFG.minimo_eventos_consecutivos
        )

        linhas.append(
            {
                "inicio_segundos": inicio,
                "fim_segundos": (
                    inicio
                    + CFG.duracao_janela
                ),
                "classe": classe,
                "confianca": float(
                    np.max(vetor)
                ),
                "evento_ativo": evento_ativo,
                "alerta_inicio": alerta_inicio,
                "alerta_ativo": alerta_ativo,
            }
        )

        classe_anterior = (
            classe
            if evento_ativo
            else None
        )

    return pd.DataFrame(linhas)

## 20. Estimar falsos alertas por hora em áudios normais

In [ ]:
arquivos_ambiente_teste = (
    metadados[
        (
            metadados["split"]
            == "teste"
        )
        & (
            metadados["classe"]
            == "ambiente_normal"
        )
    ]["arquivo"]
    .drop_duplicates()
    .tolist()
)

registros_falso_alerta = []
total_segundos = 0.0
total_alertas = 0

for arquivo in tqdm(
    arquivos_ambiente_teste,
    desc="Áudios ambientes",
):
    try:
        audio = carregar_audio(
            arquivo
        )

        timeline_ambiente = (
            inferir_audio_continuo(
                Path(arquivo)
            )
        )
    except Exception:
        continue

    duracao = (
        len(audio)
        / CFG.taxa_amostragem
    )

    alertas = int(
        timeline_ambiente[
            "alerta_inicio"
        ].sum()
    )

    total_segundos += duracao
    total_alertas += alertas

    registros_falso_alerta.append(
        {
            "arquivo": arquivo,
            "duracao_segundos": duracao,
            "alertas": alertas,
        }
    )

horas_ambiente = (
    total_segundos / 3600.0
)

falsos_alertas_hora = (
    total_alertas / horas_ambiente
    if horas_ambiente > 0
    else float("nan")
)

resumo_falsos_alertas = {
    "arquivos_ambiente": (
        len(
            registros_falso_alerta
        )
    ),
    "horas_ambiente": (
        horas_ambiente
    ),
    "falsos_alertas": (
        total_alertas
    ),
    "falsos_alertas_por_hora": (
        falsos_alertas_hora
    ),
}

print(
    json.dumps(
        resumo_falsos_alertas,
        indent=2,
        ensure_ascii=False,
    )
)

pd.DataFrame(
    registros_falso_alerta
).to_csv(
    RESULTADOS
    / "falsos_alertas_ambiente.csv",
    index=False,
)

## 21. Testar um arquivo de áudio externo

In [ ]:
from google.colab import files

enviados = files.upload()

arquivos_audio = [
    nome
    for nome in enviados
    if Path(nome).suffix.lower()
    in EXTENSOES_AUDIO
]

if len(arquivos_audio) != 1:
    raise RuntimeError(
        "Envie exatamente um arquivo de áudio."
    )

ARQUIVO_TESTE = Path(
    arquivos_audio[0]
).resolve()

print("Áudio externo:", ARQUIVO_TESTE)
display(Audio(str(ARQUIVO_TESTE)))

In [ ]:
timeline = inferir_audio_continuo(
    ARQUIVO_TESTE
)

display(
    timeline.sort_values(
        "confianca",
        ascending=False,
    ).head(20)
)

caminho_timeline = (
    RESULTADOS
    / "timeline_audio_externo.csv"
)

timeline.to_csv(
    caminho_timeline,
    index=False,
)

print(
    "Inícios de alerta:",
    int(
        timeline[
            "alerta_inicio"
        ].sum()
    ),
)

plt.figure(figsize=(14, 5))

for classe in CLASSES_EVENTO:
    parte = timeline[
        timeline["classe"] == classe
    ]

    if parte.empty:
        continue

    plt.scatter(
        parte["inicio_segundos"],
        parte["confianca"],
        label=classe,
        s=25,
    )

plt.xlabel("Tempo (s)")
plt.ylabel("Confiança")
plt.title("Linha do tempo do AED v2")
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)
plt.grid()
plt.tight_layout()
plt.show()

## 22. VAD em paralelo

O VAD identifica trechos com voz para o ramo:

```text
VAD → Whisper → NLP
```

O resultado do VAD é apenas informativo aqui. O AED acima continua processando todas as janelas.

In [ ]:
def analisar_vad(
    caminho: Path,
    agressividade: int = 2,
    frame_ms: int = 30,
) -> dict[str, Any]:
    audio = carregar_audio(caminho)
    audio = np.clip(
        audio,
        -1.0,
        1.0,
    )

    pcm = (
        audio * 32767
    ).astype(np.int16)

    amostras_frame = int(
        CFG.taxa_amostragem
        * frame_ms
        / 1000
    )

    vad = webrtcvad.Vad(
        agressividade
    )

    decisoes = []
    tempos = []

    for inicio in range(
        0,
        len(pcm)
        - amostras_frame
        + 1,
        amostras_frame,
    ):
        frame = pcm[
            inicio :
            inicio + amostras_frame
        ]

        fala = vad.is_speech(
            frame.tobytes(),
            CFG.taxa_amostragem,
        )

        decisoes.append(fala)
        tempos.append(
            inicio
            / CFG.taxa_amostragem
        )

    proporcao = (
        float(np.mean(decisoes))
        if decisoes
        else 0.0
    )

    return {
        "frames": len(decisoes),
        "frames_fala": int(
            sum(decisoes)
        ),
        "proporcao_fala": proporcao,
        "tempos": tempos,
        "decisoes": decisoes,
    }


resultado_vad = analisar_vad(
    ARQUIVO_TESTE
)

print(
    "Frames analisados:",
    resultado_vad["frames"],
)
print(
    "Frames com fala:",
    resultado_vad["frames_fala"],
)
print(
    "Proporção de fala:",
    f"{resultado_vad['proporcao_fala']:.2%}",
)

print(
    "\nInterpretação:"
    "\n- AED: analisou todo o áudio."
    "\n- VAD: indica se o Whisper deve ser acionado."
)

## 23. Gerar configuração e pacote final

In [ ]:
configuracao_exportada = {
    "versao": "aed_multiclasse_v2",
    "taxa_amostragem": CFG.taxa_amostragem,
    "duracao_janela": CFG.duracao_janela,
    "passo_janela_inferencia": (
        CFG.passo_janela_inferencia
    ),
    "n_fft": CFG.n_fft,
    "hop_length": CFG.hop_length,
    "n_mels": CFG.n_mels,
    "n_mfcc": CFG.n_mfcc,
    "fmin": CFG.fmin,
    "fmax": CFG.fmax,
    "frames": FRAMES,
    "classes": CLASSES,
    "limiares": limiares,
    "minimo_eventos_consecutivos": (
        CFG.minimo_eventos_consecutivos
    ),
    "suavizacao_janelas": (
        CFG.suavizacao_janelas
    ),
    "seleciona_janelas_com_rotulo": False,
    "normalizacao_stats": (
        normalizacao_stats
    ),
    "vad": {
        "uso": (
            "Somente para acionar "
            "Whisper/NLP"
        ),
        "bloqueia_aed": False,
        "frame_ms": 30,
        "agressividade": 2,
    },
}

(
    MODELOS / "configuracao_aed_v2.json"
).write_text(
    json.dumps(
        configuracao_exportada,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

if ENTREGA.exists():
    shutil.rmtree(ENTREGA)

ENTREGA.mkdir(parents=True)

for pasta_origem in (
    MODELOS,
    RESULTADOS,
):
    for caminho in pasta_origem.iterdir():
        if caminho.is_file():
            shutil.copy2(
                caminho,
                ENTREGA / caminho.name,
            )

resumo = {
    "configuracao": asdict(CFG),
    "classes": CLASSES,
    "limiares": limiares,
    "metricas_janela": (
        metricas_janela
    ),
    "metricas_gravacao": (
        metricas_gravacao
    ),
    "falsos_alertas": (
        resumo_falsos_alertas
    ),
    "quantidade_fontes": (
        metadados.groupby(
            ["classe", "split"]
        )["arquivo"]
        .nunique()
        .reset_index()
        .to_dict(orient="records")
    ),
}

(
    ENTREGA / "resumo_experimento.json"
).write_text(
    json.dumps(
        resumo,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

arquivo_zip = shutil.make_archive(
    "/content/aed_multiclasse_v2_entrega",
    "zip",
    root_dir=str(ENTREGA),
)

print("Pacote final:", arquivo_zip)

from google.colab import files
files.download(arquivo_zip)

## Próximas etapas no projeto

Depois de baixar o ZIP:

```text
models/audio/
├── aed_multiclasse_final.keras
├── aed_float32.tflite
├── aed_float16.tflite
├── aed_int8.tflite
├── limiares.json
└── configuracao_aed.json
```

Integração futura:

```text
INMP441
├── AED contínuo
├── VAD → faster-whisper → NLP
└── GCC-PHAT → TDOA → DOA
```

GCC-PHAT, TDOA e DOA não são treinados neste notebook; eles dependem da captura sincronizada dos microfones na Raspberry Pi.